# 3d. COMPASS Finetuner vs PreTrainer — UMAP Comparison

Сравнение UMAP проекций двух моделей **side-by-side**:
- **Left column**: `finetuner_pft_all.pt` (из `data/interim`)
- **Right column**: `pretrainer.pt` (из `data/interim_PT`)

Цель — визуально оценить:
1. Сохраняется ли биологическая структура (Diagnosis) в PreTrainer?
2. Есть ли разница в batch effects между моделями?
3. Насколько меняется пространство embeddings?

## 0. Setup

In [ ]:
!git clone https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin main)
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q --upgrade ipython ipykernel

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from pathlib import Path
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL, DIAGNOSIS_COL,
    COMPASS_UMAP_KEY,
    COMPASS_PT_UMAP_KEY,
    COHORT_CANCER_CODE,
)
from batchcor_rna_emb.data_io import load_cohort

set_logger(level="INFO")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_INTERIM_DIR    = Path('/content/drive/MyDrive/data/interim')      # Finetuner
DATA_INTERIM_PT_DIR = Path('/content/drive/MyDrive/data/interim_PT')   # PreTrainer

logger.info("Finetuner cohorts: {}", [p.name for p in sorted(DATA_INTERIM_DIR.glob('*.adata.zarr'))])
logger.info("PreTrainer cohorts: {}", [p.name for p in sorted(DATA_INTERIM_PT_DIR.glob('*.adata.zarr'))])

## 1. Plot Helpers

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.facecolor": "#f8f9fa",
    "axes.facecolor":   "#ffffff",
    "axes.edgecolor":   "#dee2e6",
    "grid.color":       "#e9ecef",
    "grid.linewidth":   0.6,
    "figure.dpi":       120,
})


def plot_umap_panel(umap_coords, color_labels, title, ax, palette=None, legend=True):
    """Scatter plot of 2D UMAP projection, colored by labels."""
    unique_labels = sorted(color_labels.unique())
    if palette is None:
        palette = sns.color_palette("husl", n_colors=len(unique_labels))
    color_map = dict(zip(unique_labels, palette))
    colors = [color_map[l] for l in color_labels]

    ax.scatter(
        umap_coords[:, 0], umap_coords[:, 1],
        c=colors, s=12, alpha=0.65, edgecolors="none", rasterized=True,
    )
    ax.set_title(title, fontsize=10, fontweight="bold", pad=8)

    if legend and len(unique_labels) <= 15:
        handles = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[l],
                   markersize=7, label=str(l))
            for l in unique_labels
        ]
        ax.legend(
            handles=handles, fontsize=6, loc="best",
            framealpha=0.85, edgecolor="#ccc",
            ncol=1 if len(unique_labels) <= 6 else 2,
        )

## 2. Per-Cohort Side-by-Side: Finetuner vs PreTrainer

Для каждой когорты — 2×2 grid:
- **Left**: Finetuner UMAP | **Right**: PreTrainer UMAP
- **Top**: colored by RNA Batch | **Bottom**: colored by Diagnosis

In [ ]:
for cohort_name in COHORT_CANCER_CODE:
    ft_path = DATA_INTERIM_DIR    / f"{cohort_name}.adata.zarr"
    pt_path = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"

    if not ft_path.exists() or not pt_path.exists():
        logger.warning("Skipping {} — missing finetuner or pretrainer data", cohort_name)
        continue

    adata_ft = load_cohort(ft_path)
    adata_pt = load_cohort(pt_path)

    umap_ft = np.asarray(adata_ft.obsm[COMPASS_UMAP_KEY])
    umap_pt = np.asarray(adata_pt.obsm[COMPASS_PT_UMAP_KEY])

    batch_ft = adata_ft.obs[BATCH_COL].astype(str)
    batch_pt = adata_pt.obs[BATCH_COL].astype(str)
    diag_ft  = adata_ft.obs[DIAGNOSIS_COL].astype(str)
    diag_pt  = adata_pt.obs[DIAGNOSIS_COL].astype(str)

    # Shared palettes for consistent colors
    all_batches = sorted(set(batch_ft.unique()) | set(batch_pt.unique()))
    all_diags   = sorted(set(diag_ft.unique())  | set(diag_pt.unique()))
    batch_pal = sns.color_palette("husl", n_colors=len(all_batches))
    diag_pal  = sns.color_palette("Set2", n_colors=len(all_diags))

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        f"{cohort_name}  ({adata_ft.n_obs} samples)\n"
        f"Finetuner (PFT)  vs  PreTrainer (PT)",
        fontsize=14, fontweight="bold", y=1.02,
    )

    # Row 0: RNA Batch
    plot_umap_panel(umap_ft[:, [0, 1]], batch_ft,
        "Finetuner — RNA Batch", axes[0, 0], palette=batch_pal)
    plot_umap_panel(umap_pt[:, [0, 1]], batch_pt,
        "PreTrainer — RNA Batch", axes[0, 1], palette=batch_pal)

    # Row 1: Diagnosis
    plot_umap_panel(umap_ft[:, [0, 1]], diag_ft,
        "Finetuner — Diagnosis", axes[1, 0], palette=diag_pal)
    plot_umap_panel(umap_pt[:, [0, 1]], diag_pt,
        "PreTrainer — Diagnosis", axes[1, 1], palette=diag_pal)

    for ax in axes.flat:
        ax.set_xlabel("UMAP-1")
        ax.set_ylabel("UMAP-2")

    plt.tight_layout()
    plt.show()

    del adata_ft, adata_pt

logger.info("Per-cohort comparison complete.")

## 3. Combined All-Cohorts Comparison

Все когорты объединены — Finetuner vs PreTrainer side-by-side,
colored by Cohort, Batch, and Diagnosis.

In [ ]:
# Collect data from both folders
ft_umaps, pt_umaps = [], []
all_cohort, all_batch, all_diag = [], [], []

for cohort_name in COHORT_CANCER_CODE:
    ft_path = DATA_INTERIM_DIR    / f"{cohort_name}.adata.zarr"
    pt_path = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"

    if not ft_path.exists() or not pt_path.exists():
        continue

    adata_ft = load_cohort(ft_path)
    adata_pt = load_cohort(pt_path)

    ft_umaps.append(np.asarray(adata_ft.obsm[COMPASS_UMAP_KEY]))
    pt_umaps.append(np.asarray(adata_pt.obsm[COMPASS_PT_UMAP_KEY]))

    n = adata_ft.n_obs
    all_cohort.extend([cohort_name] * n)
    all_batch.extend(adata_ft.obs[BATCH_COL].astype(str).tolist())
    all_diag.extend(adata_ft.obs[DIAGNOSIS_COL].astype(str).tolist())

    del adata_ft, adata_pt

umap_ft_all = np.vstack(ft_umaps)
umap_pt_all = np.vstack(pt_umaps)

df_meta = pd.DataFrame({
    "cohort":    all_cohort,
    "batch":     all_batch,
    "diagnosis": all_diag,
})

logger.info(
    "Combined: {} samples, {} cohorts, {} batches, {} diagnoses",
    len(df_meta), df_meta['cohort'].nunique(),
    df_meta['batch'].nunique(), df_meta['diagnosis'].nunique(),
)

In [ ]:
color_configs = [
    ("cohort",    "Cohort",    sns.color_palette("tab10", n_colors=df_meta['cohort'].nunique())),
    ("batch",     "RNA Batch", None),
    ("diagnosis", "Diagnosis", sns.color_palette("Set2",  n_colors=df_meta['diagnosis'].nunique())),
]

fig, axes = plt.subplots(3, 2, figsize=(18, 22))
fig.suptitle(
    f"All Cohorts Combined  ({len(df_meta)} samples)\n"
    f"Finetuner (PFT)  vs  PreTrainer (PT)",
    fontsize=15, fontweight="bold", y=1.01,
)

for row_idx, (col, label, pal) in enumerate(color_configs):
    show_legend = df_meta[col].nunique() <= 15

    plot_umap_panel(
        umap_ft_all[:, [0, 1]], df_meta[col],
        f"Finetuner — {label}", axes[row_idx, 0],
        palette=pal, legend=show_legend,
    )
    plot_umap_panel(
        umap_pt_all[:, [0, 1]], df_meta[col],
        f"PreTrainer — {label}", axes[row_idx, 1],
        palette=pal, legend=show_legend,
    )

for ax in axes.flat:
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")

plt.tight_layout()
plt.show()

logger.info("Combined comparison complete.")

## 4. Embedding Distance Summary

Количественное сравнение: насколько различаются embeddings двух моделей.

In [ ]:
from batchcor_rna_emb.config import COMPASS_EMBEDDING_KEY, COMPASS_PT_EMBEDDING_KEY

rows = []
for cohort_name in COHORT_CANCER_CODE:
    ft_path = DATA_INTERIM_DIR    / f"{cohort_name}.adata.zarr"
    pt_path = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"

    if not ft_path.exists() or not pt_path.exists():
        continue

    adata_ft = load_cohort(ft_path)
    adata_pt = load_cohort(pt_path)

    ft_emb = adata_ft.obsm[COMPASS_EMBEDDING_KEY]
    pt_emb = adata_pt.obsm[COMPASS_PT_EMBEDDING_KEY]

    corr = np.corrcoef(ft_emb.flatten(), pt_emb.flatten())[0, 1]
    mse  = np.mean((ft_emb - pt_emb) ** 2)
    cos_sims = np.array([
        np.dot(ft_emb[i], pt_emb[i]) / (np.linalg.norm(ft_emb[i]) * np.linalg.norm(pt_emb[i]) + 1e-10)
        for i in range(ft_emb.shape[0])
    ])

    rows.append({
        "Cohort": cohort_name,
        "Samples": adata_ft.n_obs,
        "Pearson r (flat)": round(corr, 4),
        "MSE": round(mse, 6),
        "Cosine sim (mean)": round(cos_sims.mean(), 4),
        "Cosine sim (std)": round(cos_sims.std(), 4),
    })

    del adata_ft, adata_pt

df_comparison = pd.DataFrame(rows)
display(df_comparison)

logger.info("Embedding comparison complete.")